In [1]:
import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')

import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre

from pathlib import Path

# Reload module to ensure using the latest version (if utils_clean.py was modified)
import importlib
import utils_clean
importlib.reload(utils_clean)
from utils_clean import (
    prepare_training_data,
    evaluate_autosort_model,
    match_neurons,
    calibration_model,
    real_time_processing,
    generate_confusion_matrix_df,
    compute_noise_detection_metrics,
    visualize_umap_features,
    SimpleAutoSort,
    SimpleWaveformLoader
)

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
sorting_new_dir = Path("/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/kilosort_spike_sorting/sorting_new")
all_dates = sorted([d.name for d in sorting_new_dir.iterdir() if d.is_dir() and d.name != '021322'])

print(f"Found {len(all_dates)} dates to process: {all_dates}")

# Set base path for results saving
base_results_dir = Path("/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/autosort_input/eval_results")
base_results_dir.mkdir(exist_ok=True)

# Other fixed paths
train_neuron_inf_path = "/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/kilosort_spike_sorting/sorting_new/021322/neuron_inf.pkl"
save_dir = "/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/autosort_input/"
model_save_dir = save_dir + "model_save/"

# Define run directories
run_dirs = ['run_1', 'run_2', 'run_3', 'run_4', 'run_5']

# Load training data neuron_inf (fixed, shared by all dates)
with open(train_neuron_inf_path, 'rb') as f:
    train_neuron_inf = pickle.load(f)

# Extract all unique tract_channels from training data neuron_inf as valid_channels
valid_channels = sorted(train_neuron_inf['tract_channel'].unique().tolist())
print(f"Number of valid channels extracted from training data neuron_inf: {len(valid_channels)}")
print(f"Valid channels list: {valid_channels}")

# Load model device (fixed, shared by all dates)
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Fixed parameters
neuron_inf_color = ["#a74a5b", "#d64158", "#e28572", "#d6522c",
                    "#a5572c", "#da9131", "#d6a46a", "#8c6d2c",
                    "#c0ab39", "#6d7821", "#9cb835", "#67733a",
                    "#9eb56c", "#4c902f", "#61c350", "#418348",
                    "#54c083", "#338b70", "#51c6c0", "#609dd8",
                    "#6365ab", "#636edd", "#a85aca", "#c590d9",
                    "#9c4d88", "#d5449a", "#e280a9"]

detection_params = {
    'thr_min': 3.5,
    'thr_max': 30,
    'distance': 3,
    'ch_max_simul_firing': 5,
    'wlen': 5,
    'prominence': 10,
}

window_params = {
    'left_sample': 10,
    'right_sample': 20,
}

calibration_duration_seconds = 120
n_additional_clusters = 10

evaluation_params = {
    'batch_size': 512,
    'left_sample': 10,
    'right_sample': 20,
}

train_neuron_list = train_neuron_inf['Neuron'].tolist()
print(f"Number of training data neurons: {len(train_neuron_inf)}")


Found 11 dates to process: ['022522', '031722', '042422', '052422', '062422', '072322', '082322', '092422', '102122', '112022', '122022']
Number of valid channels extracted from training data neuron_inf: 23
Valid channels list: [0, 3, 4, 5, 6, 8, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 26, 27]
Using device: cuda
Number of training data neurons: 27


In [3]:
all_dates = ['112022', '122022']

In [4]:
# Start loop to process all dates
from matplotlib.backends.backend_pdf import PdfPages
import umap

for date in all_dates:
    print("\n" + "=" * 80)
    print(f"Starting to process date: {date}")
    print("=" * 80)
    
    # Create results folder for current date
    date_results_dir = base_results_dir / date
    date_results_dir.mkdir(exist_ok=True)
    
    try:
        # 1. Load current date's data
        recording_path = f'/media/ubuntu/sda/data/mouse6/ns4/natural_image/mouse6_{date}_natural_image_001.ns4'
        spike_inf_path = f"/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/kilosort_spike_sorting/sorting_new/{date}/spike_inf.tsv"
        neuron_inf_path = f"/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/kilosort_spike_sorting/sorting_new/{date}/neuron_inf.pkl"
        
        # Check if files exist
        if not Path(recording_path).exists():
            print(f"Warning: recording file does not exist: {recording_path}, skipping this date")
            continue
        if not Path(spike_inf_path).exists():
            print(f"Warning: spike_inf file does not exist: {spike_inf_path}, skipping this date")
            continue
        if not Path(neuron_inf_path).exists():
            print(f"Warning: neuron_inf file does not exist: {neuron_inf_path}, skipping this date")
            continue
        
        # Load GT data
        spike_inf = pd.read_csv(spike_inf_path, sep='\t', index_col=0)
        with open(neuron_inf_path, 'rb') as f:
            neuron_inf = pickle.load(f)
        
        # Filter spike_inf: remove spikes from neurons not in valid_channels
        # First, create mapping from neuron to tract_channel
        neuron_to_tract_channel = dict(zip(neuron_inf['Neuron'], neuron_inf['tract_channel']))
        
        # Filter spike_inf: keep only spikes from neurons with tract_channel in valid_channels
        spike_inf_filtered = spike_inf[spike_inf['neuron'].isin(neuron_to_tract_channel.keys())].copy()
        spike_inf_filtered = spike_inf_filtered[
            spike_inf_filtered['neuron'].map(neuron_to_tract_channel).isin(valid_channels)
        ]
        
        original_spike_count = len(spike_inf)
        filtered_spike_count = len(spike_inf_filtered)
        removed_spike_count = original_spike_count - filtered_spike_count
        
        print(f"\nFiltering spike_inf:")
        print(f"  Original spike count: {original_spike_count}")
        print(f"  Filtered spike count: {filtered_spike_count}")
        print(f"  Removed spike count: {removed_spike_count} ({removed_spike_count/original_spike_count*100:.2f}%)")
        
        # Also filter neuron_inf: keep only neurons with tract_channel in valid_channels
        neuron_inf_filtered = neuron_inf[neuron_inf['tract_channel'].isin(valid_channels)].copy()
        original_neuron_count = len(neuron_inf)
        filtered_neuron_count = len(neuron_inf_filtered)
        removed_neuron_count = original_neuron_count - filtered_neuron_count
        
        print(f"\nFiltering neuron_inf:")
        print(f"  Original neuron count: {original_neuron_count}")
        print(f"  Filtered neuron count: {filtered_neuron_count}")
        print(f"  Removed neuron count: {removed_neuron_count} ({removed_neuron_count/original_neuron_count*100:.2f}%)")
        
        # Use filtered data
        spike_inf = spike_inf_filtered
        neuron_inf = neuron_inf_filtered
        
        # Load and preprocess recording
        recording_raw = se.read_blackrock(file_path=recording_path)
        recording_recorded = recording_raw.remove_channels(["98", '31', '32'])
        recording_f = spre.bandpass_filter(recording_recorded, freq_min=300, freq_max=3000)
        recording_f = spre.common_reference(recording_f, reference="global", operator="median")
        
        n_channels = recording_f.get_num_channels()
        print(f"Recording loaded successfully, sampling rate: {recording_f.get_sampling_frequency()} Hz, number of channels: {n_channels}")
        
        # 2. Prepare evaluation data (shared by all runs)
        eval_data_dir = Path(save_dir) / "eval_data" / date
        eval_data_dir.mkdir(parents=True, exist_ok=True)
        
        print("Preparing evaluation data...")
        duration_seconds = 200
        eval_train_data_dir = prepare_training_data(
            recording_f=recording_f,
            spike_inf=spike_inf,
            neuron_inf=neuron_inf,
            save_dir=str(eval_data_dir) + "/",
            duration_seconds=duration_seconds,
            valid_channels=valid_channels,  # Pass valid_channels parameter to detect only on valid channels
            **detection_params,
            **window_params
        )
        train_data_dir = eval_train_data_dir
        
        # 3. Neuron matching (shared by all runs)
        print("Matching neurons...")
        eval_neuron_inf_matched = match_neurons(
            train_neuron_inf=train_neuron_inf,
            eval_neuron_inf=neuron_inf,
            position_threshold=10,
            waveform_similarity_threshold=0.95
        )
        
        # Store results for all runs
        all_runs_results = []
        best_run_idx = None
        best_classification_accuracy = -1.0
        best_run_calibration_results = None
        best_run_results = None
        best_run_noise_df = None
        best_run_calib_results_df = None
        
        # 4. Loop through all runs
        print("\n" + "-" * 80)
        print(f"Evaluating all {len(run_dirs)} runs...")
        print("-" * 80)
        
        for run_idx, run_dir in enumerate(run_dirs):
            print(f"\n>>> Processing {run_dir} ({run_idx + 1}/{len(run_dirs)})...")
            
            # Load keep_id for this run
            run_model_save_dir = Path(model_save_dir) / run_dir
            run_keep_id_path = run_model_save_dir / 'keep_id.pkl'
            
            if not run_keep_id_path.exists():
                print(f"Warning: keep_id.pkl does not exist in {run_dir}, skipping this run")
                continue
            
            with open(run_keep_id_path, 'rb') as f:
                keep_id = pickle.load(f)
            
            try:
                # 4.1 Evaluate model (for calculating acc_old and acc_new)
                print(f"  Evaluating model for {run_dir}...")
                results = evaluate_autosort_model(
                    train_data_dir=train_data_dir,
                    model_save_dir=str(run_model_save_dir) + "/",
                    n_channels=n_channels,
                    **evaluation_params,
                    save_results=False,
                    eval_neuron_inf_matched=eval_neuron_inf_matched,
                    eval_data_dir=train_data_dir
                )
                
                # 4.2 Load model (prepare for calibration)
                # Create dataset to get weights
                dataset = SimpleWaveformLoader(
                    root=str(train_data_dir) + '/',
                    shank_channel=np.arange(n_channels),
                    Keep_id=keep_id
                )
                
                # Create model
                autosort_model = SimpleAutoSort(
                    ch_num=n_channels,
                    samplepoints=30,
                    device=device,
                    set_shank_id=keep_id,
                    save_dir=str(run_model_save_dir) + "/",
                    pos_weight_noise=dataset.pos_weight_noise.to(device),
                    pos_weight_label=dataset.pos_weight_label.to(device)
                )
                autosort_model.load_model()
                autosort_model.eval()
                
                # 4.3 Calibration stage
                print(f"  Starting Calibration stage for {run_dir}...")
                calibration_results = calibration_model(
                    recording_f=recording_f,
                    autosort_model=autosort_model,
                    train_neuron_inf=train_neuron_inf,
                    calibration_duration_seconds=calibration_duration_seconds,
                    n_additional_clusters=n_additional_clusters,
                    detection_params=detection_params,
                    window_params=window_params,
                    position_threshold=10.0,
                    waveform_similarity_threshold=0.9,
                    eval_neuron_inf=eval_neuron_inf_matched,
                    eval_spike_inf=spike_inf,
                    device=device,
                )
                
                # 4.4 Calculate metrics for this run
                classification_accuracy = 0.0
                acc_old = 0.0
                acc_new = 0.0
                noise_df = pd.DataFrame()
                calib_results_df = None
                
                if calibration_results['results_df'] is not None and 'gt_label' in calibration_results['results_df'].columns:
                    calib_results_df = calibration_results['results_df']
                    
                    # Generate confusion matrix
                    calib_confusion_matrix, calib_summary_df = generate_confusion_matrix_df(
                        results_df=calib_results_df,
                        train_neuron_list=train_neuron_list
                    )
                    
                    # Calculate calibration accuracy
                    if 'gt_label' in calib_summary_df.columns and 'predicted_label' in calib_summary_df.columns:
                        matched_df = calib_summary_df[
                            (calib_summary_df['gt_label'] != 'unmatch') &
                            (calib_summary_df['gt_label'] != 'noise') &
                            (calib_summary_df['predicted_label'] != 'unmatch')
                        ]
                        
                        if len(matched_df) > 0:
                            classification_accuracy = (matched_df['gt_label'] == matched_df['predicted_label']).sum() / len(matched_df)
                        else:
                            classification_accuracy = 0.0
                        
                        noise_df = calib_summary_df[calib_summary_df['gt_label'] == 'noise']
                    else:
                        classification_accuracy = 0.0
                        noise_df = pd.DataFrame()
                else:
                    print(f"  Warning: Calibration stage has no GT label data for {run_dir}")
                    classification_accuracy = 0.0
                    noise_df = pd.DataFrame()
                    calib_results_df = None
                
                # Calculate accuracy before and after adjustment for this run
                if len(noise_df) > 0 and calib_results_df is not None and len(calib_results_df) > 0:
                    prop = (noise_df['predicted_label'] == 'unmatch').sum() / len(calib_results_df)
                else:
                    prop = 0.0
                
                N = len(results['gt_noise'])
                acc_old = (results['noise_predictions'] == results['gt_noise']).mean()
                FP = ((results['noise_predictions'] == 1) & (results['gt_noise'] == 0)).sum()
                acc_new = acc_old + prop * FP / N if N > 0 else acc_old
                
                # Store results for this run
                run_result = {
                    'run': run_dir,
                    'classification_accuracy': classification_accuracy,
                    'noise_detection_accuracy': acc_old,
                    'noise_detection_accuracy_adjusted': acc_new
                }
                all_runs_results.append(run_result)
                
                print(f"  {run_dir} results:")
                print(f"    Classification accuracy: {classification_accuracy:.6f}")
                print(f"    Noise detection accuracy (before): {acc_old:.6f}")
                print(f"    Noise detection accuracy (after): {acc_new:.6f}")
                
                # Update best run if this is better
                if classification_accuracy > best_classification_accuracy:
                    best_classification_accuracy = classification_accuracy
                    best_run_idx = run_idx
                    best_run_calibration_results = calibration_results
                    best_run_results = results
                    best_run_noise_df = noise_df
                    best_run_calib_results_df = calib_results_df
                
            except Exception as e:
                print(f"  Error processing {run_dir}: {str(e)}")
                import traceback
                traceback.print_exc()
                continue
        
        # 5. Save all runs results to CSV
        if len(all_runs_results) > 0:
            all_runs_df = pd.DataFrame(all_runs_results)
            all_runs_csv_path = date_results_dir / f"all_runs_results_{date}.csv"
            all_runs_df.to_csv(all_runs_csv_path, index=False)
            print(f"\nSaved all runs results: {all_runs_csv_path}")
            print(f"Best run: {run_dirs[best_run_idx]} with classification accuracy: {best_classification_accuracy:.6f}")
        else:
            print("Warning: No runs were successfully processed")
            continue
        
        # 6. Generate and save plots using best run results
        if best_run_calib_results_df is not None and best_run_calibration_results is not None:
            # 6.1 Generate and save Confusion Matrix (from best run)
            calib_confusion_matrix, calib_summary_df = generate_confusion_matrix_df(
                results_df=best_run_calib_results_df,
                train_neuron_list=train_neuron_list
            )
            
            # Save confusion matrix CSV
            confusion_matrix_csv_path = date_results_dir / f"confusion_matrix_{date}.csv"
            calib_confusion_matrix.to_csv(confusion_matrix_csv_path)
            print(f"\nSaved confusion matrix CSV (from best run {run_dirs[best_run_idx]}): {confusion_matrix_csv_path}")
            
            # Save confusion matrix heatmap PDF
            calib_confusion_matrix_plot = calib_confusion_matrix.copy()
            if 'All' in calib_confusion_matrix_plot.index:
                calib_confusion_matrix_plot = calib_confusion_matrix_plot.drop('All')
            if 'All' in calib_confusion_matrix_plot.columns:
                calib_confusion_matrix_plot = calib_confusion_matrix_plot.drop('All', axis=1)
            
            calib_confusion_matrix_normalized = calib_confusion_matrix_plot.copy()
            column_sums = calib_confusion_matrix_normalized.sum(axis=1)
            column_sums = column_sums.replace(0, 1)
            calib_confusion_matrix_normalized = calib_confusion_matrix_normalized.div(column_sums, axis=0)
            
            fig_cm = plt.figure(figsize=(12, 10))
            sns.heatmap(
                calib_confusion_matrix_normalized,
                annot=False,
                cmap='Blues',
                cbar_kws={'label': 'Proportion'}
            )
            plt.xlabel('Predicted Label', fontsize=12)
            plt.ylabel('GT Label', fontsize=12)
            plt.tight_layout()
            
            confusion_matrix_pdf_path = date_results_dir / f"confusion_matrix_{date}.pdf"
            fig_cm.savefig(confusion_matrix_pdf_path, dpi=300, bbox_inches='tight')
            plt.close(fig_cm)
            print(f"Saved confusion matrix PDF (from best run {run_dirs[best_run_idx]}): {confusion_matrix_pdf_path}")
            
            # 6.2 Save classification_accuracy (from best run)
            if 'gt_label' in calib_summary_df.columns and 'predicted_label' in calib_summary_df.columns:
                matched_df = calib_summary_df[
                    (calib_summary_df['gt_label'] != 'unmatch') &
                    (calib_summary_df['gt_label'] != 'noise') &
                    (calib_summary_df['predicted_label'] != 'unmatch')
                ]
                
                if len(matched_df) > 0:
                    best_classification_accuracy_final = (matched_df['gt_label'] == matched_df['predicted_label']).sum() / len(matched_df)
                else:
                    best_classification_accuracy_final = 0.0
                
                accuracy_dict = {'classification_accuracy': best_classification_accuracy_final}
                accuracy_df = pd.DataFrame([accuracy_dict])
                accuracy_path = date_results_dir / f"classification_accuracy_{date}.csv"
                accuracy_df.to_csv(accuracy_path, index=False)
                print(f"Saved classification accuracy (from best run {run_dirs[best_run_idx]}): {accuracy_path}, accuracy: {best_classification_accuracy_final:.6f}")
            
            # 6.3 Save noise detection accuracy (from best run)
            if len(best_run_noise_df) > 0 and best_run_calib_results_df is not None and len(best_run_calib_results_df) > 0:
                prop = (best_run_noise_df['predicted_label'] == 'unmatch').sum() / len(best_run_calib_results_df)
            else:
                prop = 0.0
            
            N = len(best_run_results['gt_noise'])
            acc_old = (best_run_results['noise_predictions'] == best_run_results['gt_noise']).mean()
            FP = ((best_run_results['noise_predictions'] == 1) & (best_run_results['gt_noise'] == 0)).sum()
            acc_new = acc_old + prop * FP / N if N > 0 else acc_old
            
            noise_accuracy_dict = {
                'noise_detection_accuracy': acc_old,
                'noise_detection_accuracy_adjusted': acc_new
            }
            noise_accuracy_df = pd.DataFrame([noise_accuracy_dict])
            noise_accuracy_path = date_results_dir / f"noise_detection_accuracy_{date}.csv"
            noise_accuracy_df.to_csv(noise_accuracy_path, index=False)
            print(f"Saved noise detection accuracy (from best run {run_dirs[best_run_idx]}): {noise_accuracy_path}")
            print(f"  Accuracy before adjustment: {acc_old:.6f}")
            print(f"  FP count: {FP}")
            print(f"  Accuracy after adjustment: {acc_new:.6f}")
            
            # 6.4 UMAP visualization and saving (from best run)
            calib_way3_100d = best_run_calibration_results.get('way3_features_noise_100d', np.array([]))
            calib_way3_30d = best_run_calibration_results.get('way3_features_30d', np.array([]))
            calib_noise_gt_labels = best_run_calibration_results.get('noise_gt_labels', None)
            calib_noise_pred_labels = best_run_calibration_results.get('noise_pred_labels', None)
            
            # Create neuron color mapping dictionary
            neuron_color_dict = {}
            for i, neuron in enumerate(train_neuron_list):
                if i < len(neuron_inf_color):
                    neuron_color_dict[neuron] = neuron_inf_color[i]
                else:
                    cmap = plt.cm.get_cmap('tab20')
                    neuron_color_dict[neuron] = cmap(i % 20)
            
            if len(calib_way3_100d) > 0 and len(calib_way3_30d) > 0 and best_run_calib_results_df is not None:
                # Generate UMAP visualization (returns 4 figures)
                figs = visualize_umap_features(
                    way3_features_100d=calib_way3_100d,
                    way3_features_30d=calib_way3_30d,
                    results_df=best_run_calib_results_df,
                    train_neuron_list=train_neuron_list,
                    noise_gt_labels=calib_noise_gt_labels,
                    noise_pred_labels=calib_noise_pred_labels,
                    neuron_inf_color=neuron_color_dict,
                    n_samples=50000,
                    random_state=42
                )
                
                # Save UMAP PDF (four pages)
                umap_pdf_path = date_results_dir / f"umap_visualization_{date}.pdf"
                with PdfPages(umap_pdf_path) as pdf:
                    for i, fig in enumerate(figs):
                        if fig is not None:
                            pdf.savefig(fig, dpi=300, bbox_inches='tight')
                            plt.close(fig)
                print(f"Saved UMAP PDF (from best run {run_dirs[best_run_idx]}): {umap_pdf_path}")
                
                # Generate and save UMAP coordinates CSV
                # Need to recalculate UMAP coordinates to save to CSV (because visualize_umap_features internally samples)
                np.random.seed(42)
                
                # Noise Detection UMAP coordinates
                if len(calib_way3_100d) > 0:
                    from sklearn.decomposition import PCA
                    n_total_noise = len(calib_way3_100d)
                    n_sample_noise = min(50000, n_total_noise)
                    sample_indices_noise = np.random.choice(n_total_noise, n_sample_noise, replace=False)
                    way3_features_noise_sample = calib_way3_100d[sample_indices_noise]
                    
                    pca_noise = PCA(n_components=30)
                    way3_features_noise_30d = pca_noise.fit_transform(way3_features_noise_sample)
                    
                    reducer_noise = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
                    noise_umap_coords = reducer_noise.fit_transform(way3_features_noise_30d)
                    
                    noise_gt_labels_plot = calib_noise_gt_labels[sample_indices_noise] if calib_noise_gt_labels is not None and len(calib_noise_gt_labels) == n_total_noise else None
                    noise_pred_labels_plot = calib_noise_pred_labels[sample_indices_noise] if calib_noise_pred_labels is not None and len(calib_noise_pred_labels) == n_total_noise else None
                    
                    # Save noise detection UMAP CSV
                    if noise_gt_labels_plot is not None:
                        noise_detection_df_gt = pd.DataFrame({
                            'UMAP_1': noise_umap_coords[:, 0],
                            'UMAP_2': noise_umap_coords[:, 1],
                            'gt_label': ['spike' if x == 1 else 'noise' for x in noise_gt_labels_plot],
                            'predicted_label': ['spike' if x == 1 else 'noise' for x in noise_pred_labels_plot] if noise_pred_labels_plot is not None else ['unknown'] * len(noise_umap_coords)
                        })
                        noise_detection_csv_path = date_results_dir / f"umap_noise_detection_{date}.csv"
                        noise_detection_df_gt.to_csv(noise_detection_csv_path, index=False)
                        print(f"Saved noise detection UMAP CSV (from best run {run_dirs[best_run_idx]}): {noise_detection_csv_path}")
                
                # Label Classification UMAP coordinates
                if len(calib_way3_30d) > 0 and best_run_calib_results_df is not None:
                    valid_indices = []
                    valid_gt_labels = []
                    valid_pred_labels = []
                    
                    for idx in range(len(calib_way3_30d)):
                        if idx < len(best_run_calib_results_df):
                            gt_label = best_run_calib_results_df.iloc[idx]['gt_label']
                            pred_label = best_run_calib_results_df.iloc[idx]['predicted_label']
                            if (gt_label not in ['unmatch', 'noise', 'unknown', None]) and (pred_label != 'unmatch'):
                                valid_indices.append(idx)
                                valid_gt_labels.append(gt_label)
                                valid_pred_labels.append(pred_label)
                    
                    if len(valid_indices) > 0:
                        valid_indices = np.array(valid_indices)
                        way3_features_label_filtered = calib_way3_30d[valid_indices]
                        
                        n_total_label = len(way3_features_label_filtered)
                        n_sample_label = min(50000, n_total_label)
                        sample_indices_label = np.random.choice(n_total_label, n_sample_label, replace=False)
                        way3_features_label_sample = way3_features_label_filtered[sample_indices_label]
                        
                        label_gt_labels = [valid_gt_labels[i] for i in sample_indices_label]
                        label_pred_labels = [valid_pred_labels[i] for i in sample_indices_label]
                        
                        reducer_label = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
                        label_umap_coords = reducer_label.fit_transform(way3_features_label_sample)
                        
                        # Save label classification UMAP CSV
                        label_classification_df = pd.DataFrame({
                            'UMAP_1': label_umap_coords[:, 0],
                            'UMAP_2': label_umap_coords[:, 1],
                            'gt_label': label_gt_labels,
                            'predicted_label': label_pred_labels
                        })
                        label_classification_csv_path = date_results_dir / f"umap_label_classification_{date}.csv"
                        label_classification_df.to_csv(label_classification_csv_path, index=False)
                        print(f"Saved label classification UMAP CSV (from best run {run_dirs[best_run_idx]}): {label_classification_csv_path}")
            else:
                print("Warning: Calibration stage missing necessary feature data, cannot perform UMAP visualization")
        
        print(f"\nDate {date} processing completed! All results saved to: {date_results_dir}")
        
    except Exception as e:
        print(f"Error processing date {date}: {str(e)}")
        import traceback
        traceback.print_exc()
        continue

print("\n" + "=" * 80)
print("All dates processing completed!")
print("=" * 80)



Starting to process date: 112022

Filtering spike_inf:
  Original spike count: 898290
  Filtered spike count: 822539
  Removed spike count: 75751 (8.43%)

Filtering neuron_inf:
  Original neuron count: 31
  Filtered neuron count: 27
  Removed neuron count: 4 (12.90%)
Recording loaded successfully, sampling rate: 10000.0 Hz, number of channels: 30
Preparing evaluation data...
### 1. Threshold Detection
Sampling rate: 10000.0 Hz, Number of channels: 30
Recording total length: 25271176 samples (2527.12 seconds)
Will process first 2000000 samples (200.00 seconds)
Number of valid channels: 23
Valid channels list: [0, 3, 4, 5, 6, 8, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 26, 27]
Data shape: (2000000, 30)
Building detect_array...
Number of detected spikes: 343661

### 2. Load Ground Truth and Match
Building gt_array...
GT spike count: 74280
---spike detection rate: 0.9518
Number of matched spikes: 70701
Number of unmatched spikes: 272960

### 3. Extract Waveforms


Extracting waveforms: 100%|██████████| 30/30 [00:07<00:00,  4.28it/s]


Waveform extraction completed!
waveform shape: (343648, 30, 30)

### 4. Save Training Data
Save directory: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/autosort_input/eval_data/112022/train_data
  ✓ neuron_mapping.pkl saved
Saving data...
  ✓ X_waveform.pkl saved
  ✓ Y_spike_id.pkl saved
  ✓ Y_spike_id_noise.pkl saved
  ✓ X_spiketrain_time.pkl saved

All data saved to: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/autosort_input/eval_data/112022/train_data
Data statistics:
  - Total spike count: 343648
  - Number of channels: 30
  - Window length: 30
  - Number of unique units: 27
  - Noise spike count: 272948
  - Valid spike count: 70700
Matching neurons...
Neuron Matching
Matching neurons...
  Neuron_4 -> Neuron_1 (Similarity: 0.9901, Position distance: 3.33)
  Neuron_7 -> Neuron_11 (Similarity: 0.9761, Position distance: 2.48)
  Neuron_12 -> Neuron_17 (Similarity: 0.9940, Position distan

Evaluating: 100%|██████████| 672/672 [00:03<00:00, 216.37it/s]



Evaluation results (original):
  - Noise classification accuracy: 0.8490
  - Unit classification accuracy: 0.0942
  - Unit classification F1 score: 0.0942
  - Number of unit samples evaluated: 70700
  - Total samples: 343648

Computing adjusted evaluation results (treating unmatch neuron as noise)...
  - Number of unmatched neurons: 9
  - Unmatched neurons: ['Neuron_10' 'Neuron_11' 'Neuron_14' 'Neuron_16' 'Neuron_39' 'Neuron_45'
 'Neuron_48' 'Neuron_50' 'Neuron_56']
  - Number of adjusted samples: 13690

Evaluation results (adjusted):
  - Noise classification accuracy: 0.8388
  - Total samples: 343648
  - Unit classification accuracy: 0.1384
  - Unit classification F1 score: 0.0822
  - Number of unit samples evaluated: 70700
    - Matched neuron samples: 57010
    - Unmatched neuron samples: 13690
      - Correctly identified as noise: 5096 (37.2%)
      - Misclassified as unit: 8594 (62.8%)
    - Note: unmatched neuron samples are treated as noise, correctly identified as noise count

...


Evaluation results (original):
  - Noise classification accuracy: 0.8538
  - Unit classification accuracy: 0.0751
  - Unit classification F1 score: 0.0751
  - Number of unit samples evaluated: 59858
  - Total samples: 331554

Computing adjusted evaluation results (treating unmatch neuron as noise)...
  - Number of unmatched neurons: 8
  - Unmatched neurons: ['Neuron_12' 'Neuron_13' 'Neuron_16' 'Neuron_17' 'Neuron_39' 'Neuron_52'
 'Neuron_59' 'Neuron_70']
  - Number of adjusted samples: 10556

Evaluation results (adjusted):
  - Noise classification accuracy: 0.8472
  - Total samples: 331554
  - Unit classification accuracy: 0.1171
  - Unit classification F1 score: 0.0573
  - Number of unit samples evaluated: 59858
    - Matched neuron samples: 49302
    - Unmatched neuron samples: 10556
      - Correctly identified as noise: 4183 (39.6%)
      - Misclassified as unit: 6373 (60.4%)
    - Note: unmatched neuron samples are treated as noise, correctly identified as noise counts as correct

Noise classification: 100%|██████████| 491/491 [00:00<00:00, 550.92it/s]


Number of spikes passing noise classifier: 61153

### 5. PCA dimensionality reduction
Feature shape after PCA dimensionality reduction: (61153, 30)
PCA explained variance ratio: 0.8954

### 6. K-means clustering
Number of clusters: 37 (Training neurons: 27, additional: 10)
Clustering completed, number of samples per cluster:
  Cluster 0: 988 samples
  Cluster 1: 2121 samples
  Cluster 2: 3430 samples
  Cluster 3: 2489 samples
  Cluster 4: 602 samples
  Cluster 5: 1340 samples
  Cluster 6: 2982 samples
  Cluster 7: 3908 samples
  Cluster 8: 670 samples
  Cluster 9: 1958 samples
  Cluster 10: 1830 samples
  Cluster 11: 910 samples
  Cluster 12: 2964 samples
  Cluster 13: 1563 samples
  Cluster 14: 2336 samples
  Cluster 15: 1178 samples
  Cluster 16: 1840 samples
  Cluster 17: 1224 samples
  Cluster 18: 613 samples
  Cluster 19: 2020 samples
  Cluster 20: 972 samples
  Cluster 21: 1136 samples
  Cluster 22: 2288 samples
  Cluster 23: 986 samples
  Cluster 24: 1827 samples
  Cluster 25: 1

Extracting way3 features for all spikes: 100%|██████████| 491/491 [02:33<00:00,  3.20it/s]


  run_5 results:
    Classification accuracy: 0.749250
    Noise detection accuracy (before): 0.853822
    Noise detection accuracy (after): 0.863446

Saved all runs results: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/autosort_input/eval_results/122022/all_runs_results_122022.csv
Best run: run_4 with classification accuracy: 0.775386

Saved confusion matrix CSV (from best run run_4): /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/autosort_input/eval_results/122022/confusion_matrix_122022.csv
Saved confusion matrix PDF (from best run run_4): /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/autosort_input/eval_results/122022/confusion_matrix_122022.pdf
Saved classification accuracy (from best run run_4): /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/autosort_input/eval_results/122022/classification_a